# Pipeline de Ingesta: Construcción del Grafo de Conocimiento

Sistema Graph RAG sobre el canon de Sherlock Holmes.
Este notebook ejecuta el pipeline completo de ingesta:
1. Descarga de textos de Project Gutenberg
2. Separación en relatos individuales
3. Chunking consciente de la estructura
4. Extracción multipaso de entidades y relaciones
5. Entity resolution
6. Población del grafo en Neo4j

In [1]:
%load_ext autoreload
%autoreload 2
# Si ejecutas desde notebooks/, necesitas que graphrag sea importable.
# Con uv: uv sync --extra dev && pip install -e .
from graphrag.config import get_settings
from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.ingestion.text_processor import TextProcessor
from graphrag.ingestion.entity_extractor import EntityExtractor

settings = get_settings()
print(f"Proyecto GCP: {settings.google_cloud_project}")
print(f"Neo4j URI: {settings.neo4j_uri}")
print(f"Chunk size: {settings.chunk_size}")

Proyecto GCP: holmesgraphrag
Neo4j URI: bolt://localhost:7687
Chunk size: 1500


## 1. Inicializar Neo4j y crear esquema

In [2]:
neo4j = Neo4jManager()
neo4j.setup_database()  # Crea constraints + índices vectoriales + fulltext
print("Base de datos inicializada")
print(f"Stats actuales: {neo4j.get_stats()}")

Base de datos inicializada
Stats actuales: {}


## 2. Descargar y procesar textos de Gutenberg

In [3]:
processor = TextProcessor(neo4j_manager=neo4j)

#Solo los 10 relatos de desarrollo
story_chunks = processor.process_phase1()

print(f"\nRelatos procesados: {len(story_chunks)}")
for title, chunks in story_chunks.items():
    print(f"  - {title}: {len(chunks)} chunks")

Generando embeddings: 100%|██████████| 35/35 [00:00<00:00, 57.66it/s]                          
                                                                                            


Relatos procesados: 10
  - Silver Blaze: 36 chunks
  - The Final Problem: 27 chunks
  - A Scandal In Bohemia: 32 chunks
  - The Red-Headed League: 34 chunks
  - A Case Of Identity: 26 chunks
  - The Five Orange Pips: 27 chunks
  - The Adventure Of The Blue Carbuncle: 29 chunks
  - The Adventure Of The Speckled Band: 67 chunks
  - The Adventure Of The Copper Beeches: 37 chunks
  - The Adventure Of The Dancing Men: 35 chunks


## 3. Extracción de entidades y relaciones

Extracción multipaso con sliding context:
- **Paso 1**: Extracción de entidades (Characters, Locations, Crimes, Objects, Deductions, Scenes, Events)
- **Paso 2**: Extracción de relaciones entre las entidades encontradas
- **Entity Resolution**: Normalización + embeddings + LLM para desambiguar duplicados

In [4]:
'''
import logging
logging.getLogger('graphrag.ingestion.entity_extractor').setLevel(logging.DEBUG)

extractor = EntityExtractor()

all_results = {}
story_title = "A SCANDAL IN BOHEMIA"
chunks = story_chunks[story_title]

# Solo 3 chunks para debug rápido
result = extractor.process_story_chunks(chunks, story_title)
all_results[story_title] = result

entities = result["entities"]
print(f"Personajes: {len(entities.get('characters', []))}")
for c in entities["characters"]:
  print(f"  [{c['name']}] aliases: {c.get('aliases', [])}")
'''

'\nimport logging\nlogging.getLogger(\'graphrag.ingestion.entity_extractor\').setLevel(logging.DEBUG)\n\nextractor = EntityExtractor()\n\nall_results = {}\nstory_title = "A SCANDAL IN BOHEMIA"\nchunks = story_chunks[story_title]\n\n# Solo 3 chunks para debug rápido\nresult = extractor.process_story_chunks(chunks, story_title)\nall_results[story_title] = result\n\nentities = result["entities"]\nprint(f"Personajes: {len(entities.get(\'characters\', []))}")\nfor c in entities["characters"]:\n  print(f"  [{c[\'name\']}] aliases: {c.get(\'aliases\', [])}")\n'

In [5]:
extractor = EntityExtractor()

all_results = {}
for story_title, chunks in story_chunks.items():
    print(f"\n{'='*60}")
    print(f"Procesando: {story_title}")
    print(f"{'='*60}")

    result = extractor.process_story_chunks(chunks, story_title)
    all_results[story_title] = result

    # Resumen
    entities = result["entities"]
    n_chars = len(entities.get("characters", []))
    n_locs = len(entities.get("locations", []))
    n_crimes = len(entities.get("crimes", []))
    n_deductions = len(entities.get("deductions", []))
    print(f"  Personajes: {n_chars}, Ubicaciones: {n_locs}, Crímenes: {n_crimes}, Deducciones: {n_deductions}")
    print(f"  Relaciones: {len(result['relationships'])}")


Procesando: Silver Blaze


Extrayendo 'Silver Blaze':  44%|████▍     | 16/36 [10:54<14:35, 43.75s/it]model_validate_json falló, aplicando extract_json como fallback.
Error actualizando contexto en chunk 17/36: No se pudo extraer JSON de la respuesta
Extrayendo 'Silver Blaze':  58%|█████▊    | 21/36 [17:14<14:42, 58.82s/it]model_validate_json falló, aplicando extract_json como fallback.
Error actualizando contexto en chunk 22/36: No se pudo extraer JSON de la respuesta
Extrayendo 'Silver Blaze':  92%|█████████▏| 33/36 [27:46<02:23, 47.91s/it]model_validate_json falló, aplicando extract_json como fallback.
Error actualizando contexto en chunk 34/36: No se pudo extraer JSON de la respuesta
Extrayendo 'Silver Blaze':  94%|█████████▍| 34/36 [30:25<02:42, 81.02s/it]model_validate_json falló, aplicando extract_json como fallback.
Error actualizando contexto en chunk 35/36: No se pudo extraer JSON de la respuesta
Generando embeddings: 100%|██████████| 104/104 [00:03<00:00, 27.78it/s]


  Personajes: 26, Ubicaciones: 37, Crímenes: 52, Deducciones: 34
  Relaciones: 618

Procesando: The Final Problem


Generando embeddings: 100%|██████████| 85/85 [00:03<00:00, 24.49it/s]


  Personajes: 19, Ubicaciones: 21, Crímenes: 25, Deducciones: 19
  Relaciones: 411

Procesando: A Scandal In Bohemia


...

  Personajes: 30, Ubicaciones: 75, Crímenes: 56, Deducciones: 52
  Relaciones: 900

Procesando: The Adventure Of The Copper Beeches


Extrayendo 'The Adventure Of The Copper Beeches':   0%|          | 0/37 [00:00<?, ?it/s]Error transitorio (intento 1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 5s.
Generando embeddings: 100%|██████████| 103/103 [00:03<00:00, 26.81it/s]


  Personajes: 16, Ubicaciones: 34, Crímenes: 26, Deducciones: 19
  Relaciones: 542

Procesando: The Adventure Of The Dancing Men


Extrayendo 'The Adventure Of The Dancing Men':  91%|█████████▏| 32/35 [24:42<03:02, 60.87s/it]Error transitorio (intento 1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 5s.
Error transitorio (intento 1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 5s.
Generando embeddings: 100%|██████████| 107/107 [00:03<00:00, 28.85it/s]


  Personajes: 17, Ubicaciones: 29, Crímenes: 40, Deducciones: 41
  Relaciones: 609


In [6]:
import json
import os

# Guarda los resultados de extracción a disco por si el pipeline se interrumpe.
# Para recargar sin re-extraer: all_results = json.load(open("../output/extraction_results.json"))
os.makedirs("../output", exist_ok=True)
with open("../output/extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"Checkpoint guardado: ../output/extraction_results.json ({len(all_results)} relatos)")

Checkpoint guardado: ../output/extraction_results.json (10 relatos)


In [7]:
# Resolución cross-story: unifica nombres canónicos entre relatos.
# Garantiza que Holmes y Watson tengan el mismo nombre canónico en Neo4j
# independientemente del relato de origen.
all_results = extractor.normalize_cross_story_entities(all_results)

# Guarda el checkpoint normalizado (sobreescribe el anterior)
with open("../output/extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("Cross-story normalization completada.")
# Verificación rápida
for story, result in all_results.items():
    chars = result["entities"].get("characters", [])
    holmes = next((c["name"] for c in chars if "holmes" in c["name"].lower()), "--")
    watson = next((c["name"] for c in chars if "watson" in c["name"].lower()), "--")
    print(f"  {story[:40]:40s}  Holmes='{holmes}'  Watson='{watson}'")

Cross-story normalization completada.
  Silver Blaze                              Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Final Problem                         Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  A Scandal In Bohemia                      Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Red-Headed League                     Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  A Case Of Identity                        Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Five Orange Pips                      Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Adventure Of The Blue Carbuncle       Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Adventure Of The Speckled Band        Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Adventure Of The Copper Beeches       Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Adventure Of The Dancing Men          Holmes='Sherlock Holmes'  Watson='Dr. Watson'


## 4. Poblar el grafo en Neo4j

In [8]:
for story_title, result in all_results.items():
    print(f"Almacenando: {story_title}")

    # Almacenar entidades
    neo4j.store_entities(result["entities"], story_title)

    # Almacenar relaciones
    neo4j.store_relationships(result["relationships"], story_title=story_title)

    # Vincular chunks con las entidades que mencionan
    for chunk_info in result.get("chunk_entities", []):
        if chunk_info["chunk_id"] and chunk_info["entity_names"]:
            neo4j.link_chunk_to_entities(chunk_info["chunk_id"], chunk_info["entity_names"])

print("\nGrafo poblado exitosamente")

Almacenando: Silver Blaze
Almacenando: The Final Problem
Almacenando: A Scandal In Bohemia
Almacenando: The Red-Headed League
Almacenando: A Case Of Identity
Almacenando: The Five Orange Pips
Almacenando: The Adventure Of The Blue Carbuncle
Almacenando: The Adventure Of The Speckled Band
Almacenando: The Adventure Of The Copper Beeches
Almacenando: The Adventure Of The Dancing Men

Grafo poblado exitosamente


## 5. Verificar el grafo

In [9]:
stats = neo4j.get_stats()
print("Estadísticas del grafo:")
for label, count in stats.items():
    print(f"  {label}: {count}")

Estadísticas del grafo:
  Event: 615
  Object: 535
  Chunk: 350
  Deduction: 293
  Location: 252
  Crime: 218
  Scene: 212
  Character: 183
  Story: 10


In [10]:
# Ver personajes más conectados
top_characters = neo4j.execute_query("""
MATCH (c:Character)-[r]-()
RETURN c.name AS name, count(DISTINCT r) AS connections
ORDER BY connections DESC
LIMIT 10
""")

print("\nPersonajes más conectados:")
for char in top_characters:
    print(f"  {char['name']}: {char['connections']} conexiones")


Personajes más conectados:
  Sherlock Holmes: 870 conexiones
  Dr. Watson: 369 conexiones
  Mr. Jabez Wilson: 70 conexiones
  Mr. Victor Hatherley: 67 conexiones
  Mr. Hilton Cubitt: 63 conexiones
  Mr. Duncan Ross: 63 conexiones
  Miss Violet Hunter: 56 conexiones
  Mr. John Straker: 53 conexiones
  John Openshaw's father: 53 conexiones
  Dr. Grimesby Roylott: 50 conexiones


In [11]:
# Ver relatos y sus entidades
stories = neo4j.execute_query("""
MATCH (s:Story)
OPTIONAL MATCH (c:Character)-[:APPEARS_IN]->(s)
RETURN s.title AS story, s.collection AS collection, count(c) AS characters
ORDER BY characters DESC
""")

print("\nRelatos cargados:")
for s in stories:
    print(f"  {s['story']} ({s['collection']}): {s['characters']} personajes")


Relatos cargados:
  The Adventure Of The Blue Carbuncle (The Adventures of Sherlock Holmes): 31 personajes
  The Adventure Of The Speckled Band (The Adventures of Sherlock Holmes): 30 personajes
  A Scandal In Bohemia (The Adventures of Sherlock Holmes): 29 personajes
  Silver Blaze (The Memoirs of Sherlock Holmes): 26 personajes
  The Final Problem (The Memoirs of Sherlock Holmes): 19 personajes
  The Five Orange Pips (The Adventures of Sherlock Holmes): 18 personajes
  A Case Of Identity (The Adventures of Sherlock Holmes): 17 personajes
  The Adventure Of The Dancing Men (The Return of Sherlock Holmes): 17 personajes
  The Red-Headed League (The Adventures of Sherlock Holmes): 16 personajes
  The Adventure Of The Copper Beeches (The Adventures of Sherlock Holmes): 16 personajes


In [12]:
# Ver cadenas de deducción
deductions = neo4j.execute_query("""
MATCH (d:Deduction)-[:LEADS_TO]->(d2:Deduction)
RETURN d.observation AS from_obs, d2.observation AS to_obs
LIMIT 5
""")

if deductions:
    print("\nCadenas de deducción encontradas:")
    for d in deductions:
        print(f"  {d['from_obs'][:60]}... → {d['to_obs'][:60]}...")
else:
    print("\nNo se encontraron cadenas de deducción (LEADS_TO)")


Cadenas de deducción encontradas:
  The paper is peculiarly strong and stiff, not English, and h... → The 'Eg' watermark on the paper, combined with information f...
  The paper was made in Bohemia.... → The peculiar construction of the sentence—‘This account of y...
  the assistant having come for half wages... → The man’s business was a small one, and there was nothing in...
  The man’s business was a small one, and there was nothing in... → I thought of the assistant’s fondness for photography, and h...
  I thought of the assistant’s fondness for photography, and h... → I made inquiries as to this mysterious assistant and found t...


## 6. Cleanup (opcional)

In [13]:
 # Descomentar para limpiar la base de datos completa
#neo4j.clear_database()
#print("Base de datos limpiada")

neo4j.close()
print("Conexión cerrada")

Conexión cerrada
